# Research Notebook 05: Walk-Forward Cross-Validation & Statistical Testing
## Leakage-Safe Time-Series Validation and Diebold-Mariano Tests

### 1. Methodology
- **Expanding Window Validation:** Preserves causality.
- **Purging & Embargo:** For horizon $h=5$, removes boundary overlap to eliminate lookahead bias.
- **Patton (2011) QLIKE Loss:**
  $$QLIKE(h_t, \sigma_t^2) = \frac{\sigma_t^2}{h_t} - \ln\left(\frac{\sigma_t^2}{h_t}\right) - 1$$
- **Diebold-Mariano Test (1995)** with Newey-West long-run variance.


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
from volatility_lab.data.providers.synthetic import SyntheticMarketDataProvider
from volatility_lab.data.preprocessing import DataPreprocessor
from volatility_lab.features.feature_pipeline import FeaturePipeline
from volatility_lab.evaluation.walk_forward import WalkForwardValidator
from volatility_lab.evaluation.statistical_tests import diebold_mariano_test
from volatility_lab.machine_learning.registry import MLModelRegistry

raw = SyntheticMarketDataProvider(seed=42).fetch_data('NIFTY50', '2019-01-01', '2023-12-31')
df = DataPreprocessor().clean_and_prepare(raw)
X, y = FeaturePipeline().transform(df, horizon=5)

validator = WalkForwardValidator(min_train_size=400, test_size=100, step_size=100)
res_ridge = validator.evaluate_model(lambda: MLModelRegistry.get('ridge'), X, y, forecast_horizon=5)
res_lgbm = validator.evaluate_model(lambda: MLModelRegistry.get('lightgbm'), X, y, forecast_horizon=5)

dm = diebold_mariano_test(res_ridge['y_true'], res_lgbm['y_pred'], res_ridge['y_pred'], horizon=5, loss_type='qlike')
print(f"Diebold-Mariano DM Stat: {dm.statistic:.3f}, p-value: {dm.p_value:.4f}")
print(f"Superior Model: {dm.superior_model}")


### 2. Conclusions
- Standard random K-fold CV leads to severe data leakage in financial time series.
- Walk-forward testing with purging gives realistic out-of-sample error estimates.
